
# 03 - Integration Dataset
**Goal:** Define Parameters for cleaning precipitation dataset

**Outputs:**
- Final rules for the cleaning phase
- *The merged dataset is created with scripts* : `data/interim/merged_data.csv`

**Next:** Based on findings, define cleaning rules in `04_cleaning_rules_decisions.ipynb`.


### Import Libraries

In [1]:
# Standard library imports
import os
import time 
from pathlib import Path

# Third-party imports
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px

# Local application imports
from dlgoes.data.stations.clean import join_stations_thresholds
from dlgoes.data.stations.threshold import get_umbrales_resumen
from dlgoes.data.precipitation.io import read_precipitation_dataset
from dlgoes.data.precipitation.flag import simulate_qc_flags

from dlgoes.utils.config import load_config_ns
from dlgoes.utils.seed import set_seed
from dlgoes.utils.config import find_repo_root

repo_root = find_repo_root(Path.cwd())
cfg = load_config_ns(repo_root / 'configs' / 'base.yaml')
set_seed(cfg.seed)

### Read Data

In [2]:
# Read the stations dataset
df_ths = join_stations_thresholds(cfg)
stations_path = os.path.join(cfg.paths.project_root, cfg.paths.data.stations, 'stations_data.csv')
df_stations = pd.read_csv(stations_path, index_col=0, usecols=[
    'CODE', 'ESTACION', 'LON', 'LAT', 'ALT'
])
df = pd.concat([df_stations, df_ths], axis=1)
df.head()

,ESTACION,LON,LAT,ALT,THS_1,THS_2,MIN_1,MIN_2
CODE,,,,,,,,
X47E0D438,ALAMOR,-80.39788,-4.48047,116.0,9.3,10.2,0.0,0.0
X47E09732,LA ARDILLA,-80.39014,-4.48956,116.0,3.1,16.9,0.0,0.0
X472606FA,AYABACA,-79.71077,-4.63776,2633.0,5.6,11.5,0.0,0.0
X47E01126,CABO INGA H,-80.39898,-3.97972,143.0,4.4,11.9,0.0,0.0
X472F00A6,CABO INGA M,-80.40182,-3.97594,228.0,5.0,15.8,0.0,0.0


In [ ]:
# Get the thesholds summary by each station
umbrales, noMayor, errors = get_umbrales_resumen(cfg)

In [4]:
# Read precipitation data from CSV files in the raw data directory
dfPrecipitacion = read_precipitation_dataset(cfg)
dfPrecipitacion.head(2)

,CODE,NOMBRE,FECHA,HORA,PRECIPITACION,FLAG
0,X4722A338,ACJANACO,03/04/2020,00:00:00,0.0,C0000001
1,X4722A338,ACJANACO,03/04/2020,01:00:00,0.0,C0000001


In [ ]:
# DEfine the new FLAG with the provided thresholds for each station 
dfPrecipitacion['FLAG_V2'] = dfPrecipitacion.apply(lambda x: simulate_qc_flags(x, umbrales),axis=1)

In [6]:
dfPrecipitacion['FLAG'].value_counts() / len(dfPrecipitacion) * 100

FLAG
C0000001              82.563264
ND                    13.272817
C0000002               1.674404
D0230303               0.740922
D0220301               0.709880
D0230301               0.615092
M0000002               0.245205
M0000001, M0110302     0.161978
M0110302               0.015894
M0000001               0.000488
DIM00001               0.000057
Name: count, dtype: float64

In [7]:
dfPrecipitacion['FLAG_V2'].value_counts()/ len(dfPrecipitacion) * 100

FLAG_V2
C01    59.354446
NC     24.540646
ND      6.769683
M01     6.697301
D02     1.564468
D01     1.073456
Name: count, dtype: float64

In [12]:
_t = dfPrecipitacion[dfPrecipitacion['FLAG_V2']=='NC']
_t['FLAG'].value_counts() / len(_t) * 100

FLAG
C0000001    96.456992
C0000002     1.453935
D0220301     1.452298
D0230301     0.311783
D0230303     0.276478
M0110302     0.024316
M0000002     0.024199
Name: count, dtype: float64

In [8]:
df_qc_manual = dfPrecipitacion[(dfPrecipitacion['FLAG_V2'].isin(['D01', 'D02']))
                &(dfPrecipitacion['FLAG'].isin(['C0000002', 'M0000002']))]

In [9]:
df_qc_manual['FLAG'].value_counts() / len(df_qc_manual) * 100

FLAG
C0000002    98.04284
M0000002     1.95716
Name: count, dtype: float64

### Findings
- Using the SENAMHI-provided QC threshold, only 75% of the data can be classified correctly.
- Approximately 2.5% of the data (D01, D02) is suspect and must go through the manual review phase.
- Only 1.8% of the data (C01, M02) has successfully passed the manual validation phase.

### Next Actions

- Clean the dataset using the new FLAG_V2
- Match the cleaned dataset with the valid and complete GOES image dataset to generate the final development dataset.
- Recalculate the statistics after the cleaning phase.
- Move the finalized cleaning rules to `src/dlgoes/data/..`  once they are defined.
